# Session 11 — Debugging Stateful and Workflow Agents

**Halvard Works maintenance assistant.** Four agents answer one engineer's request: a planner
decides who does what, then diagnostics, documentation and maintenance do it. Today they are
followed by an **engineer's approval** and a **work order**.

> **The question for today:** *What does the next step believe — and who checked it?*

> **The sentence to take away:** *State is what the next step believes without checking.*

**You edit exactly one file today: `session-11/my_state11.py`.** Everything else is open for
you to read. Nothing in this notebook calls a model: it is free, and it gives everyone the same
output.

Halvard Works is fictional — invented for this course. Not affiliated with any vendor.

## Every term, defined before it is used

| term | in this session it means |
|---|---|
| **state** | the one dictionary every step of the workflow reads, and writes back |
| **node** | one box in the graph: a function that gets the state and returns changes to it |
| **step** | one node running once |
| **field** | one named entry in the state, like cursor or context |
| **context** | the field holding what the NEXT agent will be handed: only the previous agent's report |
| **cursor** | how many of the plan's steps have been done |
| **work order** | the paper that sends a technician to a machine with a part |
| **check** | a small function that looks at a finished run and says ok or not ok, with evidence |
| **evidence** | the one sentence a check prints to say what it saw; the only part anyone can argue with |
| **false alarm** | a check firing on a run where nothing was wrong |
| **seeded** | broken on purpose, by code we wrote, so it breaks the same way every time |
| **stub** | a fake version of the agents with no model inside: free, and identical every run |
| **status** | one word saying where the workflow is now, like AWAITING_APPROVAL |
| **status_log** | every status the workflow passed through, in order |
| **transition** | a move from one status to the next |
| **invalid transition** | a move the table of legal moves does not allow |
| **invariant** | something that must be true after every single step |
| **reducer** | a rule telling LangGraph how to combine a node's new value with the old one |
| **recursion limit** | the most steps LangGraph will run before it stops a graph with an error |
| **truncated** | stopped by a step limit before the plan was finished |
| **checkpointer** | a store that saves the state after every step so a paused graph can resume |
| **interrupt** | a pause inside a node that waits for a person's answer |
| **deadlock** | two parts each waiting for the other, so nothing ever moves |

Each term is also explained again in the cell where it first matters.

## Setup

Run this first. It loads the workflow, the broken runs and the 24 real runs. If it prints
three lines with no error, you are ready.

In [3]:
# [SETUP]
import _path  # noqa: F401  -- must be first; puts shared/ and plant/ on the path

import workflow11 as wf
import seeds11
import demos11

LIVE = seeds11.live_runs()
print('workflow11', wf.__version__)
print('real runs loaded:', len(LIVE))
print('broken runs available:', list(seeds11.BROKEN))

workflow11 s11-2026-09-28a
real runs loaded: 24
broken runs available: ['stall', 'hijack', 'counter_drift', 'skip_approval', 'order_after_reject']


## Hands-on 1 — Watch the state move (10 minutes)

The **state** is the one dictionary every step reads and writes back. A **node** is one box in
the graph — a function that gets the state and returns changes to it — and a **step** is one
node running once.

The cell below takes one **real** run (the real model answered this; nothing was broken on
purpose) and prints the state **after every step**. Read the columns:

- **cursor** — how many of the plan's steps have been done
- **steps** — how many steps have run in total; the planner counts as step 1
- **plan** — how many steps the plan has
- **context** — what the *next* agent will be handed. It is the only thing the next agent gets.

**On paper, answer:** what exactly was the **maintenance** agent handed? Was the diagnostics
report in it?

In [4]:
# [STATE_WALK]
walk = next(r for r in LIVE if r['label'] == 'live HW-001 rep1')
print('REQUEST:', walk['request'])
print()
print(wf.show_state(walk))            # one line per step
print()
print(wf.show_state(walk, full=True))  # the same steps, full text

REQUEST: The filler infeed conveyor is running hot and the vibration alarm keeps tripping. What is wrong and what do we do about it?

 # node      agent          cursor steps plan  context (what the NEXT agent receives)
 0 planner   planner             0     1    3  machine: CONVEYOR
 1 dispatch  diagnostics         1     2    3  Measured thumps per turn (3.2) ...
 2 dispatch  documentation       2     3    3  Per MAN-CONVEYOR-5.0, vibration ...
 3 dispatch  maintenance         3     4    3  The vibration trend (2.1→3.4→6.8 ...
 4 finish    finish              3     4    3  The vibration trend (2.1→3.4→6.8 ...
status_log: REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> AWAITING_APPROVAL -> APPROVED -> WORK_ORDER_ISSUED -> CLOSED

--- step 0: planner   cursor=0  steps=1  plan=3 ---
context after this step (what the NEXT agent receives):
    machine: CONVEYOR

--- step 1: diagnostics   cursor=1  steps=2  plan=3 ---
context after this step (what the NEXT agent receives):
    Measur

### Now read the full context each agent was handed

The table above shortens `context`. This cell prints it in full, step by step. Find the line
where maintenance is handed its context, and check whether the diagnostics report is in it.

In [6]:
# [CONTEXT_FULL]
for i, step in enumerate(wf.agent_steps(walk)):
    handed = wf.snapshots(walk)[i]['context']   # the state BEFORE this agent ran
    print(f'--- {step["agent"]} was handed ---')
    print(handed)
    print()

--- diagnostics was handed ---
machine: CONVEYOR

--- documentation was handed ---
Measured thumps per turn (3.2) matches the outer-race bearing signature (3.19), not the gear mesh (31.0) or inner-race (4.81). Vibration RMS is escalating rapidly (2.1 → 3.4 → 6.8 mm/s) with an accelerating trend in the last 4 days, and bearing temperature has climbed to the alarm limit of 72°C, confirming a real developing fault rather than a flat/harmless imbalance signature. Rising oil particle counts corroborate mechanical wear debris generation consistent with bearing degradation, and this is a repeat issue since the input bearing (SKF-6208) was last replaced in Sept 2025. This is a classic outer-race bearing wear signature, not cavitation (no suction data relevant) or capacitor wear (not a drive issue).

MACHINE: CONVEYOR
FAULT_CODE: BEARING-WEAR

--- maintenance was handed ---
Per MAN-CONVEYOR-5.0, vibration alarm thresholds are set at ISO 10816-3 Zone B upper limit of 4.5 mm/s RMS for this machin

## Hands-on 2 — Four work orders (20 minutes)

A **work order** is the paper that sends a technician to a machine with a part in their hand.
The system has just issued four. The cell below prints exactly what a plant manager would see.

**Vote with your partner, before you run the next cell:** for each one, would you send a
technician? Write *yes* or *no* and one reason.

In [7]:
# [FOUR_ORDERS]
VOTE = seeds11.run_vote()
for run in VOTE:
    print(wf.show(run))
    print()

== Work order 1 ==
request: The filler infeed conveyor is running hot and the vibration alarm keeps
         tripping. What is wrong and what do we do about it?
answer : 1408 characters, 7 agent steps
tail   : MACHINE=CONVEYOR, FAULT_CODE=BEARING-WEAR, SECTION=MAN-CONVEYOR-4.2, PART=SKF-6208, ACTION=replace
approval: APPROVED (SKF-6208: 2 on hand)
WORK ORDER: WO-HW-001 -> CONVEYOR, SKF-6208, replace

== Work order 2 ==
request: Doing the morning walkdown — check the filler infeed conveyor and the air-knife
         blower and tell me which needs attention first.
answer : 923 characters, 4 agent steps
tail   : MACHINE=CONVEYOR, FAULT_CODE=BEARING-WEAR, SECTION=MAN-CONVEYOR-4.2, PART=SKF-6208, ACTION=replace
approval: APPROVED (SKF-6208: 2 on hand)
WORK ORDER: WO-HW-006 -> CONVEYOR, SKF-6208, replace

== Work order 3 ==
request: Something on the Line 3 infeed is vibrating badly. Sort it out.
answer : 650 characters, 3 agent steps
tail   : MACHINE=CONVEYOR, FAULT_CODE=BEARING-WEAR, SECTIO

### The state behind each work order

Now the same four, with the state after every step and the full **status log**. Look for:

- a run where **cursor** never reaches **plan** — the plan was not finished
- a step whose context names a *different machine* from the one it was asked about
- a status log that skips a status you would expect

Change your votes if you want to. Then keep going.

In [8]:
# [FOUR_STATES]
for run in VOTE:
    print(f"=========== {run['label']} ===========")
    print(wf.show_state(run))
    print()

=========== Work order 1 ===========
 # node      agent          cursor steps plan  context (what the NEXT agent receives)
 0 planner   planner             0     1    3  machine: CONVEYOR
 1 dispatch  diagnostics         1     2    3  Compared the sensor history for ...
 2 dispatch  documentation       2     3    4  Manual section for CONVEYOR: Isolate ...
 3 dispatch  maintenance         3     4    5  Checked the store for SKF-6208: 2 on ...
 4 dispatch  maintenance         4     5    6  Checked the store for SKF-6208: 2 on ...
 5 dispatch  documentation       5     6    7  Manual section for CONVEYOR: Isolate ...
 6 dispatch  documentation       6     7    8  Manual section for CONVEYOR: Isolate ...
 7 dispatch  maintenance         7     8    9  Checked the store for SKF-6208: 2 on ...
 8 finish    finish              7     8    9  Checked the store for SKF-6208: 2 on ...
status_log: REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> RECOMMENDED -> DOCUMENTED -> DOCUMENTED -> REC

### Write two checks, then screen them

A **check** is a small function that looks at a finished run and says ok or not ok, with
**evidence**: one sentence saying what it saw. Open `session-11/my_state11.py`, read the worked
example `check_decided`, then write **`check_finished`** and **`check_machine`**.

The screener runs your checks against broken runs **and** clean ones. It prints **CATCHES IT**,
**MISSES IT**, or **FALSE ALARM** — a **false alarm** is a check firing on a run where nothing
was wrong. Some broken runs are **seeded** (broken on purpose, by code we wrote); one comes from
the **stub**, a fake version of the agents with no model inside.

Save the file, then run this cell. Run it again every time you change the file.

In [9]:
# [SCREEN_1]
import runpy, sys
sys.argv = ['screen_my_state.py', '--part', '1']
try:
    runpy.run_path(str(_path.ROOT / 'session-11' / 'screen_my_state.py'), run_name='__main__')
except SystemExit:
    pass


===== PART 1 =====

check_finished
    NOT WRITTEN

check_machine
    NOT WRITTEN


## Hands-on 3 — Approvals and work orders (20 minutes)

A **status** is one word saying where the workflow is now. The **status_log** is every status
it passed through, in order. A **transition** is one move from a status to the next.

An **invalid transition** is a move the table of legal moves (`LEGAL` in `my_state11.py`) does
not allow. An **invariant** is something that must be true after every single step — here,
`steps == cursor + 1`.

The cell below prints the status log of three clean runs and three broken ones. Before you
write anything, find the move in each broken log that should not be there.

In [11]:
# [WORKFLOW]
for rid in ('HW-001', 'HW-002', 'HW-004'):
    r = wf.run_workflow(rid)
    print(f"clean {rid:8s}", ' -> '.join(r['status_log']))
print()
for name in ('skip_approval', 'order_after_reject', 'counter_drift'):
    r = seeds11.run_broken(name)
    print(f"{name:19s}", ' -> '.join(r['status_log']))
print()
print('counter_drift, cursor and steps after every step:')
for h in wf.snapshots(seeds11.run_broken('counter_drift')):
    print(f"  {h['agent']:14s} cursor={h['cursor']}  steps={h['steps']}")

clean HW-001   REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> AWAITING_APPROVAL -> APPROVED -> WORK_ORDER_ISSUED -> CLOSED
clean HW-002   REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> AWAITING_APPROVAL -> REJECTED -> CLOSED
clean HW-004   REQUESTED -> DIAGNOSED -> CLOSED

skip_approval       REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> WORK_ORDER_ISSUED -> CLOSED
order_after_reject  REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> AWAITING_APPROVAL -> REJECTED -> WORK_ORDER_ISSUED -> CLOSED
counter_drift       REQUESTED -> DIAGNOSED -> DOCUMENTED -> RECOMMENDED -> CLOSED

counter_drift, cursor and steps after every step:
  planner        cursor=0  steps=1
  diagnostics    cursor=1  steps=2
  documentation  cursor=2  steps=4
  maintenance    cursor=3  steps=5
  finish         cursor=3  steps=5


### Fill in LEGAL, write two checks, screen them

In `my_state11.py`: fill the four empty rows of **`LEGAL`**, then write **`check_transitions`**
and **`check_counters`**. Save, then run this cell.

**Read the last part of the output.** It tells you whether your checks fired on any of the 24
**real** runs. Those are not scored, because nobody knows they are correct. If one fires,
decide with your partner: is the agent wrong, or is your check?

In [12]:
# [SCREEN_2]
import runpy, sys
sys.argv = ['screen_my_state.py', '--part', '2']
try:
    runpy.run_path(str(_path.ROOT / 'session-11' / 'screen_my_state.py'), run_name='__main__')
except SystemExit:
    pass


===== PART 2 =====

check_transitions
    NOT WRITTEN

check_counters
    NOT WRITTEN


### One real run your table probably flagged

If your `LEGAL` table is strict, it fired on this real run. Look at the order of the agents.
The engineer asked whether 88 °C is within limits. Is looking up the limit first a mistake?

In [13]:
# [REAL_FLAG]
flag = next(r for r in LIVE if r['label'] == 'live HW-012 rep1')
print(wf.show(flag)); print(); print(wf.show_state(flag))

== live HW-012 rep1 ==
request: Compressor discharge is sitting at 88 C. Is that within limits or do I need to
         act?
answer : 1460 characters, 3 agent steps
tail   : SECTION=MAN-AIR-2.2, MACHINE=AIR-COMP, FAULT_CODE=NO-FAULT, PART=, ACTION=monitor
approval: -
WORK ORDER: none

 # node      agent          cursor steps plan  context (what the NEXT agent receives)
 0 planner   planner             0     1    3  machine: AIR-COMP
 1 dispatch  documentation       1     2    3  Per **MAN-AIR-2.2 (Routine condition ...
 2 dispatch  diagnostics         2     3    3  Discharge temperature is flat across ...
 3 dispatch  maintenance         3     4    3  All monitored parameters for AIR- ...
 4 finish    finish              3     4    3  All monitored parameters for AIR- ...
status_log: REQUESTED -> DOCUMENTED -> DIAGNOSED -> RECOMMENDED -> CLOSED


## Predict, then run — what LangGraph itself does with state (8 minutes)

Three short demos. **Before each one, write down what you think it will print.**

**1. Two ways to keep a list.** A **reducer** is a rule telling LangGraph how to combine a
node's new value with the old one. Two nodes, `first` then `second`, each write their own name
into two lists. One list has no reducer. The other has `operator.add` as its reducer. What does
each list hold at the end?

In [14]:
# [REDUCER]
demos11.reducer_demo();

no_reducer   : ['second']    <- the last writer wins; 'first' is gone
with_reducer : ['first', 'second']    <- operator.add appended both


C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\langsmith\client.py:538: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e71e-95b6-7241-8e9a-bc04a4c07d2a,id=01a0e71e-95b6-7241-8e9a-bc04a4c07d2a; trace=01a0e71e-95b6-7241-8e9a-bc04a4c07d2a,id=01a0e71e-95f6-76d0-bd62-1c09bc643768; trace=01a0e71e-95b6-7241-8e9a-bc04a4c07d2a,id=01a0e71e-95f7-7093-91f4-409e5a6de1b6


**2. A loop with no way out.** One node points back at itself forever. The **recursion limit**
is the most steps LangGraph will run before it stops a graph with an error. Our four agents
have their own limit, `MAX_STEPS = 8`, and a run stopped by it is **truncated** — stopped
before the plan was finished. How many steps will LangGraph allow before it stops this one?
Guess a number, then run it (it takes about a second).

In [ ]:
# [RECURSION]
demos11.recursion_demo();

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22cb-7010-a1ab-581086168e84; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22cf-7580-8ab2-a58dd29c847b; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22cf-7580-8ab2-a58dd29c847b; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22d0-79d0-a03b-e76671de9b4b; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22d0-79d0-a03b-e76671de9b4b; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22d0-79d0-a03b-e77e2f0f0a7b; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22d0-79d0-a03b-e77e2f0f0a7b; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-22d1-7512-aca5-316e87221cd9; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id

GraphRecursionError after 8.9s: Recursion limit of 10007 reached without hitting a stop condition.
LangGraph DID stop it, and it said so loudly. But look at the number: each of those steps could have been an LLM call.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '<!doctype html><meta charset="utf-8"><meta name=viewport content="width=device-width, initial-scale=1"><title>429</title>429 Too Many Requests')trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-4397-7c62-a6d7-13499917879c; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-4397-7c62-a6d7-13499917879c; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-439c-7021-a4dd-98ba51086307; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-439c-7021-a4dd-98ba51086307; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-439d-75d3-9e60-bac45604c53a; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-439d-75d3-9e60-bac45604c53a; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-439e-7d20-85ac-3d593643d5d2; trace

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '<!doctype html><meta charset="utf-8"><meta name=viewport content="width=device-width, initial-scale=1"><title>429</title>429 Too Many Requests')trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43c9-7720-b5ae-a5f07920f2a4; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43cd-7fb2-93d7-51eb034cf360; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43cd-7fb2-93d7-51eb034cf360; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43ce-7dc1-867d-4704c3283b96; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43ce-7dc1-867d-4704c3283b96; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43cf-7013-87e9-3a86ffbd4496; trace=01a0e71f-22cb-7010-a1ab-581086168e84,id=01a0e71f-43cf-7013-87e9-3a86ffbd4496; trace

**3. Waiting for a real engineer (your instructor runs this one).** A **checkpointer** saves
the state after every step so a paused graph can resume. An **interrupt** is a pause inside a
node that waits for a person's answer. The graph below stops, is saved under a thread id, and
carries on when the engineer answers. In today's workflow the engineer is a simple rule
instead, so every laptop gets the same result for free.

In [ ]:
# [INTERRUPT_DEMO]
demos11.interrupt_demo('yes');

## Hands-on 4 — Fail loudly (12 minutes)

A check tells you *afterwards* that something went wrong. A better design stops it happening.

In `my_state11.py`, `finish_status` is asked **before anything reaches an engineer**: did the
agents finish? Right now it always says `"COMPLETE"` — which is how work order 1 got issued
even though the plan was never finished. Make it return `"STALLED"` when the plan was not
finished. A STALLED run stops: no approval, no work order.

Save, run this cell, and answer: **which work orders changed, which did not, and why not?**

In [ ]:
# [REDESIGN]
import runpy, sys
sys.argv = ['screen_my_state.py', '--part', '3']
try:
    runpy.run_path(str(_path.ROOT / 'session-11' / 'screen_my_state.py'), run_name='__main__')
except SystemExit:
    pass

## Your checks on the 24 real runs

Last cell. It runs **your** four checks on every real run and counts how often each fired.
Zero in 24 does not mean never. It means the real rate is probably under about 1 in 8 —
and, with a stub that *does* make one of these mistakes, it means nothing in the code is
stopping the real agents from making it too.

In [ ]:
# [REAL_RUNS]
import importlib, my_state11
importlib.reload(my_state11)
for name in ('check_finished', 'check_machine', 'check_transitions', 'check_counters'):
    fn = getattr(my_state11, name)
    try:
        fired = [r['label'] for r in LIVE if not fn(r)[0]]
        print(f'{name:18s} fired on {len(fired):2d} of {len(LIVE)} real runs', fired[:4])
    except NotImplementedError:
        print(f'{name:18s} not written yet')
print('work orders issued on real runs:', sum(1 for r in LIVE if r.get('work_order')))